# Panda Baseline Retrain — Kaggle Notebook A

**Purpose:** Retrain the 21M Panda model (predict mode) from scratch on the original training dataset (`GilpinLab/skew40`). This is the control run for the Koopman lifting ablation (Notebook B).

**Config:** Matches `GilpinLab/panda` (21M checkpoint) — d_model=512, 8 layers, 100k steps, `use_dynamics_embedding=True`.

**Output:** Checkpoint saved to `/kaggle/working/checkpoint-final/` — download and use locally for evaluation.

**Notebook B diff:** Change `USE_DYNAMICS_EMBEDDING = True` → `False` and `RUN_NAME = 'baseline'` → `'koopman_ablation'`. Everything else identical.

## 0. Ablation flag — only line that differs between Notebook A and B

In [ ]:
import subprocess
subprocess.run(['pip', 'uninstall', 'peft', '-y'], capture_output=True)

# Then restart kernel again (skip Cell 1 again after restart)
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
# ============================================================
# ABLATION FLAG — change this for Notebook B
# Notebook A (baseline):         USE_DYNAMICS_EMBEDDING = True
# Notebook B (Koopman ablation): USE_DYNAMICS_EMBEDDING = False
# ============================================================
USE_DYNAMICS_EMBEDDING = True
RUN_NAME = 'baseline'  # change to 'koopman_ablation' for Notebook B

## 1. Install dependencies

In [ ]:
# Install panda repo (architecture + training code)
!git clone --depth=1 https://github.com/abao1999/panda.git

# Install panda dependencies
# Note: panda uses uv but we install manually for Kaggle compatibility
%cd panda
!pip install -e . --quiet

# Additional dependencies needed for training
!pip install gluonts wandb --quiet

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
        print(f'  VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')
        print(f'  Compute capability: {torch.cuda.get_device_capability(i)}')

## 2. Load training data from HuggingFace

In [ ]:
from datasets import load_dataset
import numpy as np

print('Downloading GilpinLab/skew40...')
hf_dataset = load_dataset('GilpinLab/skew40', split='train')
print(f'Loaded {len(hf_dataset)} trajectories')
print(f'Columns: {hf_dataset.column_names}')
example = hf_dataset[0]
target = np.array(example['target'])
print(f'Example trajectory shape: {target.shape}')
print(f'Start: {example["start"]}')

In [ ]:
from gluonts.dataset.common import Dataset as GluonTSDataset
import pandas as pd
import numpy as np

print('Converting to pandas...')
df = hf_dataset.to_pandas()
print(f'Done. Shape: {df.shape}')

print('Extracting trajectories...')
targets = []
for i, row in df.iterrows():
    target = np.array(row['target'], dtype=np.float32)
    shape = row['target._np_shape']
    if shape is not None:
        target = target.reshape(shape)
    targets.append(target)

starts = df['start'].tolist()
print(f'Loaded {len(targets)} trajectories')
print(f'Sample shape: {targets[0].shape}')
print(f'Approx RAM: {sum(t.nbytes for t in targets) / 1e9:.2f} GB')

class InMemoryGluonDataset(GluonTSDataset):
    def __init__(self, targets, starts, freq='h'):
        self.targets = targets
        self.starts = starts
        self.freq = freq

    def __iter__(self):
        for target, start in zip(self.targets, self.starts):
            yield {
                'start': pd.Period(start, freq=self.freq),
                'target': target
            }

    def __len__(self):
        return len(self.targets)

test_item = next(iter(InMemoryGluonDataset(targets, starts)))
print(f'Wrapper test — shape: {test_item["target"].shape}, start: {test_item["start"]}')
print('Pre-load OK')

## 3. Model config — 21M baseline

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/panda')

from transformers import PatchTSTConfig
from panda.patchtst.patchtst import PatchTSTForPrediction
from panda.utils.train_utils import load_patchtst_model

MODEL_CONFIG = dict(
    mode='predict',
    context_length=512,
    prediction_length=128,
    patch_length=16,
    patch_stride=16,
    num_hidden_layers=8,
    d_model=512,
    num_attention_heads=8,
    channel_attention=True,
    ffn_dim=512,
    norm_type='rmsnorm',
    norm_eps=1e-5,
    attention_dropout=0.0,
    positional_dropout=0.0,
    path_dropout=0.0,
    ff_dropout=0.0,
    bias=True,
    activation_function='gelu',
    pre_norm=True,
    use_cls_token=False,
    init_std=0.02,
    scaling='std',
    pooling_type='max',
    head_dropout=0.0,
    channel_rope=False,
    max_wavelength=500,
    rope_percent=0.75,
    loss='mse',
    distribution_output=None,
    use_dynamics_embedding=USE_DYNAMICS_EMBEDDING,
    num_poly_feats=120,
    poly_degrees=2,
    rff_trainable=False,
    rff_scale=1.0,
    num_rff=256,
    do_mask_input=None,
    mask_type='random',
    random_mask_ratio=0.5,
    channel_consistent_masking=False,
    mask_value=0,
    num_forecast_mask_patches=3,
    unmasked_channel_indices=None,
    num_parallel_samples=100,
)

model = load_patchtst_model(
    mode='predict',
    model_config=MODEL_CONFIG,
    pretrained_encoder_path=None,
    pretained_checkpoint=None,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: use_dynamics_embedding={USE_DYNAMICS_EMBEDDING}')
print(f'Trainable parameters: {trainable_params:,}')

## 4. Build training dataset

In [ ]:
import torch
from torch.utils.data import Dataset as TorchDataset
import numpy as np
import sys
sys.path.insert(0, '/kaggle/working/panda')

from panda.augmentations import (
    RandomTakensEmbedding,
    RandomConvexCombinationTransform,
    RandomAffineTransform,
    StandardizeTransform,
    RandomDimSelectionTransform,
)

SEED = 99

# Instantiate augmentations matching dataset.yaml
# probabilities [1.0, 1.0, 1.0, 0.0, 0.0] — first three active
aug_takens = RandomTakensEmbedding(lag_range=(1, 10), random_seed=SEED)
aug_convex = RandomConvexCombinationTransform(alpha=1.0, dim_range=(3, 8), random_seed=SEED)
aug_affine = RandomAffineTransform(scale=1.0, dim_range=(3, 8), random_seed=SEED)
standardize = StandardizeTransform()
dim_select = RandomDimSelectionTransform(num_dims=3, random_seed=SEED)

AUGMENTATION_RATE = 0.2  # from dataset.yaml

class PandaTrainDataset(TorchDataset):
    def __init__(self, targets, context_len=512, pred_len=128,
                 n_windows_per_traj=10, fixed_dim=3, seed=99):
        self.context_len = context_len
        self.pred_len = pred_len
        self.fixed_dim = fixed_dim
        self.rng = np.random.default_rng(seed)

        self.windows = []
        window_len = context_len + pred_len

        print('Pre-sampling windows with augmentations...')
        for i, target in enumerate(targets):
            C, T = target.shape
            if T < window_len:
                continue

            # Step 1: standardise full trajectory (StandardizeTransform)
            target = standardize(target, axis=-1)

            # Step 2: apply one augmentation with probability AUGMENTATION_RATE
            if self.rng.random() < AUGMENTATION_RATE:
                aug_idx = self.rng.integers(0, 3)  # choose one of three active augs
                try:
                    if aug_idx == 0:
                        target = aug_takens(target)
                    elif aug_idx == 1:
                        target = aug_convex(target)
                    else:
                        target = aug_affine(target)
                    # re-standardise after augmentation
                    target = standardize(target, axis=-1)
                except Exception:
                    pass  # if augmentation fails, use original

            # Step 3: select fixed_dim channels
            C_aug = target.shape[0]
            if C_aug > fixed_dim:
                ch_idx = self.rng.choice(C_aug, size=fixed_dim, replace=False)
                target = target[ch_idx]
            elif C_aug < fixed_dim:
                # pad by repeating channels if needed
                repeats = (fixed_dim // C_aug) + 1
                target = np.tile(target, (repeats, 1))[:fixed_dim]

            C_final, T_final = target.shape
            if T_final < window_len:
                continue

            # Step 4: sample windows
            max_start = T_final - window_len
            starts = np.linspace(0, max_start, n_windows_per_traj, dtype=int)
            for s in starts:
                self.windows.append(target[:, s:s+window_len].astype(np.float32))

            if (i + 1) % 5000 == 0:
                print(f'  Processed {i+1}/{len(targets)} trajectories...')

        self.windows = np.array(self.windows, dtype=np.float32)
        print(f'Pre-sampled {len(self.windows)} windows, shape: {self.windows.shape}')

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        window = self.windows[idx]  # [C, context+pred], already standardised
        ctx = window[:, :self.context_len]
        mean = ctx.mean(axis=-1, keepdims=True)
        std = ctx.std(axis=-1, keepdims=True).clip(min=1e-4)
        window_norm = ((window - mean) / std).clip(-5, 5)

        past = window_norm[:, :self.context_len].T
        future = window_norm[:, self.context_len:].T

        return {
            'past_values': torch.tensor(past, dtype=torch.float32),
            'future_values': torch.tensor(future, dtype=torch.float32),
        }

train_dataset = PandaTrainDataset(
    targets,
    context_len=512,
    pred_len=128,
    n_windows_per_traj=10,
    fixed_dim=3,
    seed=SEED
)

# Smoke test
sample = train_dataset[0]
print(f'past_values: {sample["past_values"].shape}')
print(f'future_values: {sample["future_values"].shape}')
print('Dataset OK')

In [ ]:
import torch, time

model.cuda()
model.train()

# Dummy batch matching our training shape
past = torch.randn(32, 512, 3).cuda().half()
future = torch.randn(32, 128, 3).cuda().half()

# Warmup
with torch.autocast('cuda'):
    for _ in range(3):
        out = model(past_values=past, future_values=future)
        out.loss.backward()

# Time 10 steps
torch.cuda.synchronize()
t0 = time.time()
for _ in range(10):
    model.zero_grad()
    with torch.autocast('cuda'):
        out = model(past_values=past, future_values=future)
        out.loss.backward()
torch.cuda.synchronize()
elapsed = time.time() - t0
print(f'Pure forward+backward: {elapsed/10*1000:.1f} ms/step')
print(f'Theoretical max throughput: {1/(elapsed/10):.1f} it/s')

## 5. Training

In [ ]:
# Diagnostic — check for pathological windows
vals = train_dataset.windows
print(f'Windows shape: {vals.shape}')
print(f'Global min: {vals.min():.4f}')
print(f'Global max: {vals.max():.4f}')
print(f'Global mean: {vals.mean():.4f}')
print(f'Global std: {vals.std():.4f}')

# Check for windows with extreme values
abs_max_per_window = np.abs(vals).max(axis=(1,2))
print(f'\nPer-window abs max:')
print(f'  median: {np.median(abs_max_per_window):.4f}')
print(f'  95th pct: {np.percentile(abs_max_per_window, 95):.4f}')
print(f'  99th pct: {np.percentile(abs_max_per_window, 99):.4f}')
print(f'  max: {abs_max_per_window.max():.4f}')
print(f'  Windows with abs_max > 10: {(abs_max_per_window > 10).sum()}')
print(f'  Windows with abs_max > 100: {(abs_max_per_window > 100).sum()}')

In [ ]:
# Find which specific windows cause large losses
model.eval()
device = torch.device('cuda')
model = model.to(device)

from torch.utils.data import DataLoader, Subset
import torch

# Use same seed as training to get same batch order
torch.manual_seed(99)
diag_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True,
    generator=torch.Generator().manual_seed(99)
)

diag_iter = iter(diag_loader)
for step_idx in range(1, 10):
    batch = next(diag_iter)
    past = batch['past_values'].to(device)
    future = batch['future_values'].to(device)
    with torch.no_grad():
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(past_values=past, future_values=future)
    print(f'batch {step_idx}: loss={out.loss.item():.4f}, '
          f'past_max={past.abs().max().item():.4f}, '
          f'pred_max={out.prediction_outputs.abs().max().item():.4f}')

In [ ]:
import torch
import os, json, time
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup

OUTPUT_DIR = f'/kaggle/working/{RUN_NAME}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_STEPS = 50000       # change to 50_000 for full run
BATCH_SIZE = 256
LR = 1e-4
WARMUP_RATIO = 0.2
LOG_EVERY = 500        # change to 500 for full run
SAVE_EVERY = 10000       # change to 10_000 for full run
GRAD_CLIP = 1.0
SEED = 99

torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.0)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(MAX_STEPS * WARMUP_RATIO),
    num_training_steps=MAX_STEPS,
)

scaler = torch.cuda.amp.GradScaler(init_scale=256)
model.train()

step = 0
data_iter = iter(loader)
losses = []

print(f'Run: {RUN_NAME}')
print(f'use_dynamics_embedding: {USE_DYNAMICS_EMBEDDING}')
print(f'Batch: {BATCH_SIZE}, Steps: {MAX_STEPS}, Device: {device}')
print('Starting...\n')

t_start = time.time()
t_log = time.time()

while step < MAX_STEPS:
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        batch = next(data_iter)

    past = batch['past_values'].to(device)
    future = batch['future_values'].to(device)

    optimizer.zero_grad()
    with torch.autocast('cuda', dtype=torch.float16):
        out = model(past_values=past, future_values=future)
        loss = out.loss

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    losses.append(loss.item())
    step += 1

    if step % LOG_EVERY == 0:
        elapsed = time.time() - t_log
        avg_loss = sum(losses[-LOG_EVERY:]) / LOG_EVERY
        its = LOG_EVERY / elapsed
        remaining = (MAX_STEPS - step) / its / 3600
        print(f'step {step:>6}/{MAX_STEPS} | loss {avg_loss:.4f} | '
              f'{its:.2f} it/s | ~{remaining:.1f}h remaining')
        t_log = time.time()

    if step % SAVE_EVERY == 0:
        ckpt_dir = os.path.join(OUTPUT_DIR, f'checkpoint-{step}')
        model.save_pretrained(ckpt_dir)
        print(f'  Saved checkpoint: {ckpt_dir}')

final_dir = os.path.join(OUTPUT_DIR, 'checkpoint-final')
model.save_pretrained(final_dir)
with open(os.path.join(final_dir, 'training_info.json'), 'w') as f:
    json.dump({
        'run_name': RUN_NAME,
        'use_dynamics_embedding': USE_DYNAMICS_EMBEDDING,
        'max_steps': MAX_STEPS,
        'model_config': MODEL_CONFIG,
    }, f, indent=2)

total_time = (time.time() - t_start) / 3600
print(f'\nDone. Total time: {total_time:.2f}h')
print(f'Final checkpoint: {final_dir}')

## 6. Quick in-distribution sanity check

Before downloading, verify the checkpoint produces sensible forecasts on one held-out trajectory from the test split. This is not the full evaluation — just a sanity check that training converged.

In [ ]:
final_ckpt_dir = f'/kaggle/working/baseline/checkpoint-final'

In [ ]:
from datasets import load_dataset as hf_load
from panda.patchtst.pipeline import PatchTSTPipeline
import numpy as np

# Load test split
hf_test = hf_load('GilpinLab/skew40', split='test')
test_example = hf_test[0]
target = np.array(test_example['target'])
if 'target._np_shape' in test_example and test_example['target._np_shape'] is not None:
    target = target.reshape(test_example['target._np_shape'])

# target shape: [C, T] — take first 512 as context
C, T = target.shape
context = target[:, :512].T  # [512, C] — pipeline expects [T, C]
context_tensor = torch.tensor(context, dtype=torch.float32)

# Load from saved checkpoint
pipeline = PatchTSTPipeline.from_pretrained(
    mode='predict',
    pretrain_path=final_ckpt_dir,
    device_map='cpu',
)

pred = pipeline.predict(context_tensor, 128, limit_prediction_length=False)
pred = pred.squeeze().cpu().numpy()

# Ground truth for comparison
truth = target[:, 512:640].T  # [128, C]

# Compute MAE as quick sanity metric
# Per-window normalise (same as our evaluation protocol)
ctx_mean = context.mean(axis=0, keepdims=True)
ctx_std = context.std(axis=0, keepdims=True) + 1e-8
pred_norm = (pred - ctx_mean) / ctx_std
truth_norm = (truth - ctx_mean) / ctx_std
mae = np.mean(np.abs(pred_norm - truth_norm))

print(f'Sanity check MAE (normalised, one test trajectory): {mae:.4f}')
print('If MAE < 1.0, training likely converged. If >> 1.0, check training logs.')